In [1]:
import DataManipulation as DM
import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats 
import importlib
import warnings
import math
import copy
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [2]:
otb = pd.read_csv('oneTarBaseTask.csv')
etb = pd.read_csv('eightTarBaseTask.csv')
otssm = pd.read_csv('oneTarSSMTask.csv')
etssm = pd.read_csv('eightTarSSMTask.csv')
dfs = [otb,etb,otssm,etssm]

In [3]:
                  
demog = pd.read_csv("demogPaper.csv")
tmpAll = pd.concat([dfs[i] for i in range(4)], ignore_index=True)
tmpAll = tmpAll.merge(demog, on='participant', how='left')
ages = []
pps = tmpAll['participant'].unique()
print(len(pps))
for p in pps:
    try:
        i = tmpAll[tmpAll['participant']==p]['age'].iloc[0]
        age = int(i)
        ages.append(age)
    except ValueError:
        pass
print(len(ages))
print(np.nanmean(ages))
print(np.nanstd(ages))

761
755
34.11788079470199
11.75559658792815


In [4]:


tmpAll = pd.concat([dfs[i] for i in range(4)], ignore_index=True)
tmpAll = tmpAll.merge(demog, on='participant', how='left')
sexes = []
pps = tmpAll['participant'].unique()
for p in pps:
    try:
        i = tmpAll[tmpAll['participant']==p]['sex'].iloc[0]
        sex = int(i)
        if sex == 2 or sex == 1:
            sexes.append(sex)
    except ValueError:
        pass
print(len(sexes))
numMales = 0
numFemales = 0
for i in sexes:
    if i == 1:
        numMales += 1
    elif i == 2:
        numFemales += 1
       
print(numMales)
print(numFemales)

756
316
440


In [5]:
for df in dfs:
    df['participantNum'] = np.arange(len(df))

In [6]:
                                                                     
listCols = [] 
it=0
for df in dfs:
    colsToManip = [i for i in df.columns if (pd.api.types.is_object_dtype(df[i]) and i != 'participant')]
    listCols.append(colsToManip)
                               
    df = DM.dfColumnManip(df,DM.colFloater,colsToManip)
    dfs[it] = df
    it+=1

In [7]:
                                                                 
                                                                           
                          
def deep_copy_df_with_nested(df):
    """
    Creates a true deep copy of a DataFrame, including recursive deep-copy of nested lists/dicts in object columns.
    """
                                 
    df_copy = df.copy(deep=True)
                                                                
    for col in df_copy.columns:
        if df_copy[col].dtype == 'object':
                                                                      
            df_copy[col] = df_copy[col].apply(copy.deepcopy)
    return df_copy
                                                         
dfsIncOutliers = [deep_copy_df_with_nested(df) for df in dfs]
                                
excludeTrials = True
if excludeTrials:
    importlib.reload(DM)
    ATFloor=0.1                                                                        
    iqrLimit=3000                                               
    numParts=[24,10,24,10]                                                                                  
    keepFirst=5                                                                            
    ignore=['rt']                                                                                        
    nanTs,varNames,threshes,pChan,atNans,metNans = [],[],[],[],[],[]
    for i in range(len(dfs)):
                                                                                                                                                                                                  
                                             
        df,nanIdcs,metrics,thresholds,pertChanges,atIdcs,metIdcs=DM.nanOutlierTrials(dfs[i],listCols[i],ATFloor=ATFloor,iqrLimit=iqrLimit,numParts=numParts[i],keepFirst=keepFirst,ignore=listCols+ignore)
        df.drop(df.columns[df.columns.str.contains('Unnamed')], axis=1, inplace=True)
        dfs[i] = df
        nanTs.append(nanIdcs)
        varNames.append(metrics)
        threshes.append(thresholds)
        pChan.append(pertChanges)
        atNans.append(atIdcs)
        metNans.append(metIdcs)                        


In [8]:
                   
                      
                       

importlib.reload(DM)
medATFloor=0.1
madErrCeil=90
propNanCeil=0.2                                              
rems = []
medAts = []
madErrs = []
propNans = []
badDfs = [[]] * (len(dfs))

for i in range(len(dfs)):
    rem,at,err,nan = DM.outlierPPIDs(dfs[i],medATFloor=medATFloor,madErrCeil=madErrCeil,propNanCeil=propNanCeil)
    rems.append(rem)
    medAts.append(at)
    madErrs.append(err)
    propNans.append(nan)

keepAllTrials = True
if keepAllTrials:
    dfs = dfsIncOutliers                                                             
    dfsIncOutliers = [deep_copy_df_with_nested(df) for df in dfs]
    
for i in range(len(dfs)):
    rem = rems[i]
    dfs[i],badDfs[i] = DM.removeIndices(dfs[i],rem)
    print(len(dfsIncOutliers[i]))
    print(len(dfs[i]))

                                                      
                                                                                            
                                                                                                   

num exluded because zero mean aim MAGNITUDE: 0
num exluded because too many nans: 3
num exluded because too high MAD error: 0
AT 0
num exluded because zero mean aim MAGNITUDE: 0
num exluded because too many nans: 19
num exluded because too high MAD error: 0
AT 1
num exluded because zero mean aim MAGNITUDE: 1
num exluded because too many nans: 0
num exluded because too high MAD error: 0
AT 0
num exluded because zero mean aim MAGNITUDE: 3
num exluded because too many nans: 27
num exluded because too high MAD error: 0
AT 0
98
95
301
281
61
60
301
271


In [9]:
if not keepAllTrials:
    total_at_per_group = []
    grand_total_at = 0
    grand_total_trials = 0
    for i in range(len(dfs)):
        rem_set = set(rems[i])
        group_total_at = 0
        group_total_trials = 0
        orig_num_pp_for_at = len(atNans[i])                                                                 
        kept_df = dfs[i]
        kept_pp_count = len(kept_df)
        orig_num_pp = len(dfsIncOutliers[i])                                
        num_excluded = len(rems[i])
        
                                                                          
        for pp_idx in range(orig_num_pp_for_at):
            pp_num = dfsIncOutliers[i]['participantNum'].iloc[pp_idx]
            if pp_num not in rem_set:
                group_total_at += len(atNans[i][pp_idx])
        
                                                                                                  
        skipped_count = 0
        for j in range(kept_pp_count):
            aim_data = kept_df['aim'].iloc[j]
            if isinstance(aim_data, list):
                group_total_trials += len(aim_data)
            else:
                skipped_count += 1
                pp_num = kept_df['participantNum'].iloc[j]
                print(f"Warning: Skipping kept participant {pp_num} in group {i} due to invalid 'aim' data.")
        
        if skipped_count > 0:
            print(f"Group {i}: Skipped {skipped_count} kept participants due to invalid 'aim' data.")
        
        percentage = (group_total_at / group_total_trials * 100) if group_total_trials > 0 else 0
        total_at_per_group.append(group_total_at)
        grand_total_at += group_total_at
        grand_total_trials += group_total_trials
        print(f"Group {i}: {group_total_at} AT-outlier trials ({percentage:.2f}%) out of {group_total_trials} total trials across {kept_pp_count} kept participants "
              f"(originally {orig_num_pp} participants, {num_excluded} excluded)")
    
    overall_percentage = (grand_total_at / grand_total_trials * 100) if grand_total_trials > 0 else 0
    print(f"Grand total: {grand_total_at} AT-outlier trials ({overall_percentage:.2f}%) out of {grand_total_trials} total trials across all kept participants")

In [10]:
                                                                       
if excludeTrials:
    badAtNans = []
    badMetNans = []
    badNTs = []
    badThreshes = []
    for i in range(len(rems)):
        tn = []
        tt = []
        ban = []
        bmn = []
        for r in reversed(rems[i]):
            tn.append(nanTs[i].pop(r))
            tt.append(threshes[i].pop(r))
            ban.append(atNans[i].pop(r))
            bmn.append([metNans[i][m].pop(r) for m in range(len(metrics))])
        badNTs.append(tn)
        badThreshes.append(tt)
        badAtNans.append(ban)
        badMetNans.append(bmn)
                                                                                           
    badMetNans = [[[[trial for trial in pp[met]] for pp in list(reversed(group))] for group in badMetNans] for met in range(len(metrics))]
                   
    badAtNans = [list(reversed(i)) for i in badAtNans]
    badNTs = [list(reversed(i)) for i in badNTs]
    badThreshes = [[j[i] for i in np.arange(len(j)-1,-1,-1)] for j in badThreshes]

In [11]:
eLens = []
for i in range(len(dfs)):
    el = len(dfs[i]['aim'].iloc[0])
    eLens.append(el)

In [12]:
                                          
listCols = [] 
it=0
for df in dfs:
                       
    DM.addTrialNums(dfs[it])
                                             
    colsToManip = [i for i in df.columns if (pd.api.types.is_object_dtype(df[i]) and i != 'participant')]
    listCols.append(colsToManip)
                      
    dfs[it] = df.explode(listCols[it])
    it+=1


In [13]:
                                                                                                
                                           
importlib.reload(DM)
blockSizes = [60,400,60,400]
blLength = [15,40,15,40]
for i in range(4):
    DM.addBlockNum(df=dfs[i],newColName='blockNum',secSize=blockSizes[i],expLen=eLens[i])
    DM.addSignFlipMarker(df=dfs[i],newColName='flip',secSize=blockSizes[i],expLen=eLens[i])
    DM.expFlipSign(df=dfs[i],chgVar=['aim','imp','error','rotation','rawaim','rawimp','rawerror'],coeff=-1,condVar='flip',condVal=1)
    DM.addBlockTrial(df=dfs[i],newColName='blockTrial',secSize=blockSizes[i],expLen=eLens[i],blLength=blLength[i])
                                                                              
    DM.addBlockRot(df=dfs[i],newColName='blockRot',secSize=blockSizes[i],expLen=eLens[i])
    
                  

In [14]:

                  
                                          
listCols = [] 
it=0
for df in dfsIncOutliers:
                       
    DM.addTrialNums(dfsIncOutliers[it])
                                             
    colsToManip = [i for i in df.columns if (pd.api.types.is_object_dtype(df[i]) and i != 'participant')]
    listCols.append(colsToManip)
                      
    dfsIncOutliers[it] = df.explode(listCols[it])
    it+=1

In [15]:
                  
                                                                                                
                                           
importlib.reload(DM)
blockSizes = [60,400,60,400]
blLength = [15,40,15,40]
for i in range(4):
    DM.addBlockNum(df=dfsIncOutliers[i],newColName='blockNum',secSize=blockSizes[i],expLen=eLens[i])
    DM.addSignFlipMarker(df=dfsIncOutliers[i],newColName='flip',secSize=blockSizes[i],expLen=eLens[i])
    DM.expFlipSign(df=dfsIncOutliers[i],chgVar=['aim','imp','error','rotation','rawaim','rawimp','rawerror'],coeff=-1,condVar='flip',condVal=1)
    DM.addBlockTrial(df=dfsIncOutliers[i],newColName='blockTrial',secSize=blockSizes[i],expLen=eLens[i],blLength=blLength[i])
                                                                              
    DM.addBlockRot(df=dfsIncOutliers[i],newColName='blockRot',secSize=blockSizes[i],expLen=eLens[i])

In [16]:
importlib.reload(DM)
colours = ['#44AA99'] + ['#88CCEE'] + ['#FF9825'] + ['#CC6677'] + ['#AA4499']
otb = dfs[0]
etb = dfs[1]
otssm = dfs[2]
etssm = dfs[3]
dfs = [otb,etb,otssm,etssm]
DM.addCycleColumn(dfs[1])
DM.addCycleColumn(dfs[3])
[DM.addTotalAngle(df) for df in dfs if 'imp' in df.columns]
BTDat = pd.read_csv('BTDat.csv')
                                                
DM.addPhaseIndicator(dfs[0], phaseLengths=[15,30,15])
DM.addPhaseIndicator(dfs[1], phaseLengths=[40,320,40])
DM.addPhaseIndicator(dfs[2], phaseLengths=[15,30,15])
DM.addPhaseIndicator(dfs[3], phaseLengths=[40,320,40])
DM.addBlockNum(BTDat,400,400)
DM.addPhaseIndicator(BTDat, phaseLengths=[40,320,40])
DM.blockDeNanRotation(dfs[0],[15,45,60])
DM.blockDeNanRotation(dfs[1],[40,320,400])
DM.blockDeNanRotation(dfs[2],[15,45,60])
DM.blockDeNanRotation(dfs[3],[40,320,400])
DM.blockDeNanRotation(BTDat,[40,320,400])

cg = dfs[3]
cg['experiment'] = 'cg'
BTDat['experiment'] = 'bt'
BTDat['participantNum']+=1000
cgbtDF = pd.concat([cg,BTDat])

dfs[0]['targetCount'] = 1
dfs[1]['targetCount'] = 8
dfs[2]['targetCount'] = 1
dfs[3]['targetCount'] = 8
dfs[0]['hasAutocorrection'] = 0
dfs[1]['hasAutocorrection'] = 0
dfs[2]['hasAutocorrection'] = 1
dfs[3]['hasAutocorrection'] = 1
BTDat['targetCount'] = 8

6904
13777
25401
28200
28361
28803
60300
3378
3652
4124
4709
7342
13783
1691
1883
7705
10504
16664
17680
18946
25008
25045
29102
29384
32232
34879
39791
41144
42526
43703
49240
56040
67071
70489
72543
75465
76580
76865
78101
79942
80931
80989
83745
89680
94060
96275
101544


In [17]:
baseDfs = pd.concat(dfs)
baseDfs['imp'] = baseDfs['imp'].fillna(0)
baseDfs['totalAngle'] = baseDfs['aim'] + baseDfs['imp']

In [18]:
baseDfs.to_csv("CGData_.csv", index=False)
BTDat.to_csv("BTData_.csv", index=False)

In [19]:

importlib.reload(DM)



otb = dfsIncOutliers[0]
etb = dfsIncOutliers[1]
otssm = dfsIncOutliers[2]
etssm = dfsIncOutliers[3]
dfs = [otb,etb,otssm,etssm]
                                                          
for df in dfs:
    if 'participantNum' in df.columns and 'aim' in df.columns:
        nanCounts = df.groupby('participantNum')['aim'].apply(lambda x: x.isna().sum())
        badParticipants = nanCounts[nanCounts > 100].index.tolist()
        if badParticipants:
            dropIndex = df[df['participantNum'].isin(badParticipants)].index
            df.drop(dropIndex, inplace=True)
DM.addCycleColumn(dfs[1])
DM.addCycleColumn(dfs[3])
[DM.addTotalAngle(df) for df in dfs if 'imp' in df.columns]

                                                
DM.addPhaseIndicator(dfs[0], phaseLengths=[15,30,15])
DM.addPhaseIndicator(dfs[1], phaseLengths=[40,320,40])
DM.addPhaseIndicator(dfs[2], phaseLengths=[15,30,15])
DM.addPhaseIndicator(dfs[3], phaseLengths=[40,320,40])
DM.blockDeNanRotation(dfs[0],[15,45,60])
DM.blockDeNanRotation(dfs[1],[40,320,400])
DM.blockDeNanRotation(dfs[2],[15,45,60])
DM.blockDeNanRotation(dfs[3],[40,320,400])
DM.blockDeNanRotation(BTDat,[40,320,400])

cg = dfs[3]
cg['experiment'] = 'cg'

3172
7264
14137
26121
28920
29081
29523
33990
63900
3378
3652
4124
4709
7342
14143
1691
1883
2211
2332
5388
5498
8505
11304
17464
18480
19746
25808
25845
30302
30584
33832
36091
36879
41791
43144
45326
46903
54040
60840
68182
72271
76089
78543
81465
83380
83665
85301
87142
88131
88189
91345
97680
103260
105875
112344
113332


In [20]:
dfs[0]['targetCount'] = 1
dfs[1]['targetCount'] = 8
dfs[2]['targetCount'] = 1
dfs[3]['targetCount'] = 8
dfs[0]['hasAutocorrection'] = 0
dfs[1]['hasAutocorrection'] = 0
dfs[2]['hasAutocorrection'] = 1
dfs[3]['hasAutocorrection'] = 1
baseDfs = pd.concat(dfs) 
baseDfs['imp'] = baseDfs['imp'].fillna(0)
baseDfs['totalAngle'] = baseDfs['aim'] + baseDfs['imp']

In [21]:
baseDfs.to_csv("CGDataWithOutliers_.csv", index=False)

In [22]:
len(dfs[1]['participant'].unique())

301